<a href="https://colab.research.google.com/github/alisikandarriaz/lithium_supply_chain_plausibility/blob/main/lithium_supply_chain_plausibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lithium Supply Chain Plausibility Assessment

A geospatial research pipeline assessing transparency and reliability in the global lithium supply chain using real open data, spatial analysis, and knowledge graphs.

When a company declares a lithium supply chain path, can we verify it is geologically consistent, geographically feasible, and structurally complete?

**Three plausibility rules:**
- Rule 1 — Geological consistency
- Rule 2 — Geographic feasibility
- Rule 3 — Network structure completeness

In [ ]:
# ============================================================
# SECTION 1 — SETUP
# ============================================================

!pip install pandas numpy matplotlib folium -q
!pip install networkx pyvis pdfplumber requests -q
!pip install geopandas openpyxl -q
!pip install gdown -q
!pip install earthengine-api geemap -q

print("All libraries installed successfully")

---
## Section 2 — Connect to Google Drive and Verify Data

Connecting to Google Drive and verifying all downloaded data files are present before any analysis begins.

In [ ]:
# ============================================================
# SECTION 2 — Data Download + Output Setup
# ============================================================
# Part A: Downloads 7 data files from public Google Drive
#         into this Colab session's temporary storage
#         (/content/data/) — no manual download needed
#
# Part B: Mounts YOUR OWN Google Drive so all output
#         files (charts, maps, reports) are saved
#         permanently to your Drive
# ============================================================

import os
import gdown
import requests
from google.colab import drive

print("=" * 60)
print("SECTION 2 — Data Download + Output Setup")
print("=" * 60)
print()

# ── PART A: Create folder structure ─────────────────────────
print("Creating folders in Colab session...")
print()

for folder in [
    '/content/data/01_geological',
    '/content/data/03_trade_flows',
]:
    os.makedirs(folder, exist_ok=True)
    print(f"  Created: {folder}")

print()

# ── PART A: Download the 5 MRDS CSV files ───────────────────
# These are regular Google Drive files — use gdown

print("Downloading MRDS geological files...")
print()

mrds_files = [
    {
        'name': '01_mrds_greenbushes.csv',
        'id':   '104gapJIrFSH8QqTjagnleVS4vhHg35Np',
    },
    {
        'name': '01b_mrds_atacama.csv',
        'id':   '1_fXa8oo5F0fAdlyUKn0syPNssM6RGaJ5',
    },
    {
        'name': '01c_mrds_qinghai.csv',
        'id':   '1IEac1MmPO1-W9OwnnhDC1UV7_GBZNj_O',
    },
    {
        'name': '01d_mrds_wodgina.csv',
        'id':   '1nVVNF6J2nxL9izvqsnKrArhwo7dw2jw_',
    },
    {
        'name': '01e_mrds_pilgangoora.csv',
        'id':   '1QP59g7ZE6OkmhG2iBlC-Nivmud8NDdct',
    },
]

all_ok = True

for f in mrds_files:
    output_path = '/content/data/01_geological/' + f['name']
    url = f"https://drive.google.com/uc?id={f['id']}"
    try:
        gdown.download(url, output_path, quiet=True)
        size = os.path.getsize(output_path)
        print(f"  OK    {f['name']}  ({size:,} bytes)")
    except Exception as e:
        print(f"  FAIL  {f['name']}  — {e}")
        all_ok = False

print()

# ── PART A: Download GA Critical Minerals Excel ──────────────
# This is a Google Sheets file — needs export URL not gdown
# We export it directly as XLSX format

print("Downloading GA Critical Minerals 2025...")
print()

ga_sheet_id  = '1dDk3e8dh8luk8zlj6qulFcu-Xn-tmdyq'
ga_export_url = f'https://docs.google.com/spreadsheets/d/{ga_sheet_id}/export?format=xlsx'
ga_output_path = '/content/data/01_geological/03b_ga_critical_minerals_2025.xlsx'

try:
    response = requests.get(ga_export_url)
    if response.status_code == 200:
        with open(ga_output_path, 'wb') as f:
            f.write(response.content)
        size = os.path.getsize(ga_output_path)
        print(f"  OK    03b_ga_critical_minerals_2025.xlsx  ({size:,} bytes)")
    else:
        print(f"  FAIL  GA Critical Minerals — HTTP {response.status_code}")
        print(f"        Make sure the Google Sheet is set to")
        print(f"        Anyone with the link can view")
        all_ok = False
except Exception as e:
    print(f"  FAIL  GA Critical Minerals — {e}")
    all_ok = False

print()

# ── PART A: Download Comtrade CSV ────────────────────────────
print("Downloading UN Comtrade trade flow data...")
print()

comtrade_id   = '1rp9BD_sBU67-o2KHKAwsiCjZNFbFjhRB'
comtrade_path = '/content/data/03_trade_flows/09_comtrade_lithium_flows.csv'
comtrade_url  = f"https://drive.google.com/uc?id={comtrade_id}"

try:
    gdown.download(comtrade_url, comtrade_path, quiet=True)
    size = os.path.getsize(comtrade_path)
    print(f"  OK    09_comtrade_lithium_flows.csv  ({size:,} bytes)")
except Exception as e:
    print(f"  FAIL  09_comtrade_lithium_flows.csv  — {e}")
    all_ok = False

print()

# ── PART A: Verify all 7 files ───────────────────────────────
print("Verifying all downloads...")
print()

all_files = [
    '/content/data/01_geological/01_mrds_greenbushes.csv',
    '/content/data/01_geological/01b_mrds_atacama.csv',
    '/content/data/01_geological/01c_mrds_qinghai.csv',
    '/content/data/01_geological/01d_mrds_wodgina.csv',
    '/content/data/01_geological/01e_mrds_pilgangoora.csv',
    '/content/data/01_geological/03b_ga_critical_minerals_2025.xlsx',
    '/content/data/03_trade_flows/09_comtrade_lithium_flows.csv',
]

for path in all_files:
    name = os.path.basename(path)
    if os.path.exists(path):
        size = os.path.getsize(path)
        if size > 100:
            print(f"  PASS  {name}  ({size:,} bytes)")
        else:
            print(f"  FAIL  {name}  — file too small, may be corrupt")
            all_ok = False
    else:
        print(f"  FAIL  {name}  — file not found")
        all_ok = False

print()

# ── PART A: Set path variables for all other sections ────────
BASE_GEOLOGICAL = '/content/data/01_geological/'
BASE_TRADE      = '/content/data/03_trade_flows/'

# Individual file paths — use these in Section 3
PATH_GREENBUSHES = BASE_GEOLOGICAL + '01_mrds_greenbushes.csv'
PATH_ATACAMA     = BASE_GEOLOGICAL + '01b_mrds_atacama.csv'
PATH_QINGHAI     = BASE_GEOLOGICAL + '01c_mrds_qinghai.csv'
PATH_WODGINA     = BASE_GEOLOGICAL + '01d_mrds_wodgina.csv'
PATH_PILGANGOORA = BASE_GEOLOGICAL + '01e_mrds_pilgangoora.csv'
PATH_GA_EXCEL    = BASE_GEOLOGICAL + '03b_ga_critical_minerals_2025.xlsx'
PATH_COMTRADE    = BASE_TRADE      + '09_comtrade_lithium_flows.csv'

# ── PART B: Mount student's own Google Drive ─────────────────
print("=" * 60)
print("PART B — Mounting YOUR Google Drive for outputs")
print("=" * 60)
print()
print("A permission popup will appear.")
print("Click ALLOW to save your output files permanently.")
print()

drive.mount('/content/drive')

# Create output folder in student's own Drive
OUTPUT_FOLDER = '/content/drive/MyDrive/lithium-plausibility-outputs/'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print()
print(f"  Output folder ready:")
print(f"  My Drive → lithium-plausibility-outputs/")
print()

# ── Final status message ─────────────────────────────────────
if all_ok:
    print("=" * 60)
    print("SECTION 2 COMPLETE — Everything is ready")
    print()
    print("  Data files : downloaded to Colab session")
    print("  Outputs    : will save to your Google Drive")
    print("               My Drive/lithium-plausibility-outputs/")
    print()
    print("  Run Section 3 to continue.")
    print("=" * 60)
    print("NOTE: In Section 8 (Remote Sensing) you will be asked")
    print("to sign in to Google Earth Engine. Use the SAME")
    print("Google account you just used for Drive above.")
else:
    print("=" * 60)
    print("WARNING — One or more files failed to download")
    print()
    print("  Fix: Check each file in Google Drive is set to")
    print("       Anyone with the link can VIEW")
    print("       (not Restricted, not Editor)")
    print()
    print("  For the GA Sheets file specifically:")
    print("       File → Share → Anyone with link → Viewer")
    print("=" * 60)

---
## Section 3 — Data Loading

Loading all structured data directly from downloaded source files. Every coordinate and value comes from a verified source.

**Sources:**
- USGS MRDS CSV files → mine coordinates and geology
- Geoscience Australia Excel → Australian mine locations and status
- UN Comtrade CSV → lithium trade flow volumes
- OpenStreetMap Nominatim API → processing plant and manufacturer coordinates

In [ ]:
# ============================================================
# SECTION 3 — STEPS 1, 2, 3
# Load all data files using paths from Section 2
# ============================================================

import pandas as pd
import requests
import time

print("=" * 60)
print("SECTION 3 — Data Loading")
print("=" * 60)

# ------------------------------------------------------------
# STEP 1 — Load MRDS geological files
# Source: USGS Mineral Resources Data System
# Files downloaded in Section 2 to /content/data/
# ------------------------------------------------------------

print()
print("STEP 1 — Loading MRDS files...")
print()

mrds = {}

mrds_files = {
    'Greenbushes': PATH_GREENBUSHES,
    'Atacama':     PATH_ATACAMA,
    'Qinghai':     PATH_QINGHAI,
    'Wodgina':     PATH_WODGINA,
    'Pilgangoora': PATH_PILGANGOORA,
}

for name, path in mrds_files.items():
    try:
        df = pd.read_csv(
            path,
            sep='\t',
            on_bad_lines='skip',
            low_memory=False
        )
        mrds[name] = df
        print(f"  OK    {name}: {len(df)} records loaded")
    except Exception as e:
        print(f"  FAIL  {name}: {e}")

print()
print(f"  Total MRDS datasets loaded: {len(mrds)}")

# ------------------------------------------------------------
# STEP 2 — Load Geoscience Australia Critical Minerals 2025
# Source: Geoscience Australia (CC BY 4.0)
# dx.doi.org/10.26186/150802
# ------------------------------------------------------------

print()
print("STEP 2 — Loading Geoscience Australia data...")
print()

try:
    ga_full = pd.read_excel(
        PATH_GA_EXCEL,
        sheet_name='CM Mines and Deposits 2025',
        header=12
    )

    # Filter for lithium mines only
    ga_li = ga_full[
        ga_full['Commodity Group'].str.contains(
            'Lithium', na=False, case=False)
    ].copy()

    print(f"  OK    GA full dataset:     {len(ga_full)} sites")
    print(f"  OK    GA lithium filtered: {len(ga_li)} sites")
    print()

    # Show the Australian lithium mines we need
    au_mines = ['Greenbushes', 'Wodgina', 'Pilgangoora']
    print("  Checking Australian mine coordinates:")
    for mine in au_mines:
        row = ga_li[ga_li['Name'] == mine]
        if len(row) > 0:
            lat = row['Latitude'].values[0]
            lon = row['Longitude'].values[0]
            status = row['Status'].values[0]
            print(f"    FOUND  {mine}: {lat}, {lon} — {status}")
        else:
            print(f"    MISSING  {mine} — not in GA dataset")

except Exception as e:
    print(f"  FAIL — {e}")
    print("  Check that the GA Excel sheet name is correct:")
    print("  Expected: 'CM Mines and Deposits 2025'")

# ------------------------------------------------------------
# STEP 3 — Load UN Comtrade trade flow data
# Source: UN Comtrade HS Code 282520
# Lithium oxide and hydroxide exports 2022-2024
# reporters: Australia, Chile, China
# ------------------------------------------------------------

print()
print("STEP 3 — Loading UN Comtrade data...")
print()

try:
    comtrade_raw = pd.read_csv(PATH_COMTRADE)

    print(f"  OK    Raw rows: {len(comtrade_raw)}")
    print(f"  OK    Columns:  {list(comtrade_raw.columns)}")
    print()

    # Column mapping — Comtrade columns are shifted
    # Correct mapping confirmed from data inspection:
    comtrade = pd.DataFrame({
        'country': comtrade_raw['reporterISO'],
        'year':    comtrade_raw['refPeriodId'],
        'qty_kg':  comtrade_raw['qtyUnitAbbr'],
        'fob_usd': comtrade_raw['fobvalue'],
    })

    print("  Trade data by country and year:")
    print()
    summary = comtrade.groupby(
        ['country', 'year']
    ).agg(
        qty_kg=('qty_kg', 'first'),
        fob_usd=('fob_usd', 'first')
    ).reset_index()

    print(summary.to_string(index=False))

except Exception as e:
    print(f"  FAIL — {e}")

print()
print("=" * 60)
print("Steps 1, 2, 3 complete")
print("ga_li, mrds, and comtrade variables are ready")
print("Proceed to Steps 4, 5, 6 below")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 3 — STEPS 4, 5, 6
# ============================================================

# ------------------------------------------------------------
# STEP 4 — Extract mine coordinates from source files
# ------------------------------------------------------------

print("=" * 60)
print("STEP 4 — Mine Coordinates from Source Files")
print("=" * 60)

def ga_coords(name):
    row = ga_li[ga_li['Name'] == name]
    if len(row) > 0:
        return (float(row['Latitude'].values[0]),
                float(row['Longitude'].values[0]),
                str(row['Status'].values[0]))
    return None, None, None

# Australian mines from GA Critical Minerals 2025
gb_lat, gb_lon, gb_st = ga_coords('Greenbushes')
wd_lat, wd_lon, wd_st = ga_coords('Wodgina')
pg_lat, pg_lon, pg_st = ga_coords('Pilgangoora')

# Atacama from MRDS — lithium producer record
at_df   = mrds['Atacama']
at_li   = at_df[at_df['commod1'].str.contains(
    'Lithium', na=False, case=False)]
at_prod = at_li[at_li['dev_stat'].str.contains(
    'Producer|Plant', na=False)]
at_row  = at_prod.iloc[0]
at_lat  = float(at_row['latitude'])
at_lon  = float(at_row['longitude'])

# Qinghai from MRDS — Da Chaidan Lake
# Confirmed Producer in Qinghai province
qh_df  = mrds['Qinghai']
qh_row = qh_df[
    qh_df['site_name'] == 'Da Chaidan Lake'
].iloc[0]
qh_lat = float(qh_row['latitude'])
qh_lon = float(qh_row['longitude'])

print(f"Greenbushes  (GA 2025): {gb_lat}, {gb_lon}")
print(f"Wodgina      (GA 2025): {wd_lat}, {wd_lon}")
print(f"Pilgangoora  (GA 2025): {pg_lat}, {pg_lon}")
print(f"Atacama      (MRDS):    {at_lat}, {at_lon}")
print(f"Qinghai      (MRDS):    {qh_lat}, {qh_lon}")

# ------------------------------------------------------------
# STEP 5 — Geocode processing plants and manufacturers
# Source: OpenStreetMap Nominatim
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("STEP 5 — Geocoding via OpenStreetMap Nominatim")
print("Source: nominatim.openstreetmap.org")
print("=" * 60)

def geocode(query):
    url = "https://nominatim.openstreetmap.org/search"
    params = {'q': query, 'format': 'json', 'limit': 1}
    headers = {'User-Agent': 'LithiumPlausibility/1.0'}
    try:
        r = requests.get(
            url, params=params,
            headers=headers, timeout=10)
        d = r.json()
        if d:
            return float(d[0]['lat']), float(d[0]['lon'])
        return None, None
    except:
        return None, None

facilities = {
    'Kwinana Plant':
        'Kwinana, Western Australia, Australia',
    'La Negra Refinery':
        'La Negra, Antofagasta, Chile',
    'Kemerton Plant':
        'Kemerton, Western Australia, Australia',
    'Ganfeng Refinery':
        'Xinyu, Jiangxi, China',
    'CATL':
        'Ningde, Fujian, China',
    'LG Chem':
        'Cheongju, Chungcheongbuk-do, South Korea',
    'Panasonic':
        'Osaka, Japan',
    'BYD':
        'Shenzhen, Guangdong, China',
    'Samsung SDI':
        'Yongin, Gyeonggi-do, South Korea'
}

geo = {}
for name, query in facilities.items():
    lat, lon = geocode(query)
    geo[name] = {'lat': lat, 'lon': lon}
    status = '✅' if lat else '⚠️'
    print(f"{status} {name}: {lat}, {lon}")
    time.sleep(1)

print(f"\nGeocoded: "
      f"{sum(1 for v in geo.values() if v['lat'])}"
      f"/{len(facilities)} facilities")

# ------------------------------------------------------------
# STEP 6 — Build Master Supply Chain Table
# Relationships manually curated from source PDFs
# Every value cited to its source document below
#
# Path 1: IGO Annual Report 2024, p.31
# Path 2: SQM Annual Report 2024, p.18
# Path 3: Albemarle Annual Report 2025, p.22
# Path 4: Ganfeng Annual Report 2024, p.15, p.28
# Path 5: Pilbara Minerals Annual Report 2024, p.19
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("STEP 6 — Master Supply Chain Table")
print("=" * 60)

def g(name):
    return geo[name]['lat'], geo[name]['lon']

kw_lat, kw_lon = g('Kwinana Plant')
ln_lat, ln_lon = g('La Negra Refinery')
km_lat, km_lon = g('Kemerton Plant')
gf_lat, gf_lon = g('Ganfeng Refinery')
ct_lat, ct_lon = g('CATL')
lg_lat, lg_lon = g('LG Chem')
pa_lat, pa_lon = g('Panasonic')
bd_lat, bd_lon = g('BYD')
sm_lat, sm_lon = g('Samsung SDI')

master = pd.DataFrame({
    'path_id': [1,1,1,2,2,2,3,3,3,4,4,4,5,5,5],
    'tier': [
        'Mining','Processing','Manufacturing',
        'Mining','Processing','Manufacturing',
        'Mining','Processing','Manufacturing',
        'Mining','Processing','Manufacturing',
        'Mining','Processing','Manufacturing'
    ],
    'node_name': [
        # IGO Annual Report 2024, p.31
        'Greenbushes Mine','Kwinana Plant','CATL',
        # SQM Annual Report 2024, p.18
        'Salar de Atacama','La Negra Refinery','LG Chem',
        # Albemarle Annual Report 2025, p.22
        'Wodgina Mine','Kemerton Plant','Panasonic',
        # Ganfeng Annual Report 2024, p.15, p.28
        'Qinghai Lake Brine','Ganfeng Refinery','BYD',
        # Pilbara Minerals Annual Report 2024, p.19
        'Pilgangoora Mine','Chinese Refineries','Samsung SDI'
    ],
    'company': [
        'Talison (Tianqi 51%+Albemarle 49%)',  # IGO 2024 p.8
        'Tianqi Lithium Energy Australia',      # IGO 2024 p.31
        'Contemporary Amperex Technology',      # CATL ESG 2025
        'SQM + Albemarle',                      # SQM 2024 p.5
        'Albemarle',                            # SQM 2024 p.18
        'LG Chem',                              # LGChem 2024
        'MARBL JV (Albemarle 60%+MinRes 40%)', # Albemarle 2025 p.22
        'Albemarle',                            # Albemarle 2025 p.22
        'Panasonic Energy',                     # CATL ESG 2025
        'Ganfeng Lithium',                      # Ganfeng 2024 p.15
        'Ganfeng Lithium',                      # Ganfeng 2024 p.15
        'BYD Company',                          # Ganfeng 2024 p.28
        'Pilbara Minerals',                     # Pilbara 2024 p.19
        'Multiple Chinese refineries',          # USGS MCS 2026
        'Samsung SDI'                           # SamsungSDI 2025
    ],
    'country': [
        'Australia','Australia','China',        # IGO 2024
        'Chile','Chile','South Korea',          # SQM 2024
        'Australia','Australia','Japan',        # Albemarle 2025
        'China','China','China',                # Ganfeng 2024
        'Australia','China','South Korea'       # Pilbara 2024
    ],
    'latitude': [
        gb_lat,kw_lat,ct_lat,
        at_lat,ln_lat,lg_lat,
        wd_lat,km_lat,pa_lat,
        qh_lat,gf_lat,bd_lat,
        pg_lat,gf_lat,sm_lat
    ],
    'longitude': [
        gb_lon,kw_lon,ct_lon,
        at_lon,ln_lon,lg_lon,
        wd_lon,km_lon,pa_lon,
        qh_lon,gf_lon,bd_lon,
        pg_lon,gf_lon,sm_lon
    ],
    'coord_source': [
        'GA Critical Minerals 2025',
        'OpenStreetMap Nominatim',
        'OpenStreetMap Nominatim',
        'USGS MRDS',
        'OpenStreetMap Nominatim',
        'OpenStreetMap Nominatim',
        'GA Critical Minerals 2025',
        'OpenStreetMap Nominatim',
        'OpenStreetMap Nominatim',
        'USGS MRDS',
        'OpenStreetMap Nominatim',
        'OpenStreetMap Nominatim',
        'GA Critical Minerals 2025',
        'OpenStreetMap Nominatim',
        'OpenStreetMap Nominatim'
    ],
    'source_document': [
        'IGO Annual Report 2024',
        'IGO Annual Report 2024',
        'CATL ESG Report 2025',
        'SQM Annual Report 2024',
        'SQM Annual Report 2024',
        'LG Chem Sustainability Report 2024',
        'Albemarle Annual Report 2025',
        'Albemarle Annual Report 2025',
        'CATL ESG Report 2025',
        'Ganfeng Annual Report 2024',
        'Ganfeng Annual Report 2024',
        'Ganfeng Annual Report 2024',
        'Pilbara Minerals Annual Report 2024',
        'USGS MCS 2026',
        'Samsung SDI Sustainability Report 2025'
    ]
})

print(f"Total nodes: {len(master)}")
print(f"Total paths: {master['path_id'].nunique()}")
print(f"\nFull table:")
print(master[[
    'path_id','tier','node_name',
    'country','source_document'
]].to_string(index=False))

print("\n")
print("=" * 60)
print("Section 3 complete")
print("All values cited to source documents")
print("=" * 60)

---
## Section 4 — Trade Flow Charts

Visualizing lithium export data loaded from UN Comtrade CSV file. Three charts showing export volume, price collapse, and 2024 market comparison across Australia, Chile, and China.

In [ ]:
# ============================================================
# SECTION 4 — TRADE FLOW CHARTS
# Data source: UN Comtrade HS Code 282520
# ============================================================

import matplotlib.pyplot as plt

# ── Use comtrade from Section 3 ──────────────────────────────
trade = comtrade.copy()

# Calculate price per kg — needed for Chart 2
trade['price_per_kg'] = (
    trade['fob_usd'] / trade['qty_kg']
).round(2)

# ── Color per country ────────────────────────────────────────
colors = {
    'Australia': '#2196F3',
    'Chile':     '#4CAF50',
    'China':     '#F44336'
}

countries = trade['country'].unique().tolist()
years     = sorted(trade['year'].unique().tolist())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Global Lithium Supply Chain — Trade Flow Analysis\n'
    'Source: UN Comtrade HS Code 282520',
    fontsize=13, fontweight='bold'
)

# ── CHART 1 — Export volume by country and year ──────────────
ax1   = axes[0]
width = 0.25
x     = range(len(years))

for i, country in enumerate(countries):
    vals = []
    for yr in years:
        row = trade[
            (trade['country'] == country) &
            (trade['year'] == yr)
        ]
        vals.append(
            row['qty_kg'].values[0] / 1e6
            if len(row) > 0 else 0
        )
    ax1.bar(
        [xi + i * width for xi in x],
        vals, width,
        label=country,
        color=colors[country],
        alpha=0.85
    )

ax1.set_xlabel('Year')
ax1.set_ylabel('Export Volume (million kg)')
ax1.set_title('Lithium Export Volume by Country')
ax1.set_xticks([xi + width for xi in x])
ax1.set_xticklabels(years)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# ── CHART 2 — Price per kg over time ─────────────────────────
ax2 = axes[1]

for country in countries:
    subset = trade[
        trade['country'] == country
    ].sort_values('year')
    ax2.plot(
        subset['year'],
        subset['price_per_kg'],
        marker='o',
        linewidth=2.5,
        label=country,
        color=colors[country]
    )
    for _, row in subset.iterrows():
        ax2.annotate(
            f"${row['price_per_kg']}",
            (row['year'], row['price_per_kg']),
            textcoords='offset points',
            xytext=(0, 8),
            fontsize=8,
            ha='center'
        )

ax2.set_xlabel('Year')
ax2.set_ylabel('Price per kg (USD)')
ax2.set_title('Lithium Price Collapse 2022-2024')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_xticks(years)

# ── CHART 3 — Latest year export volume comparison ───────────
ax3 = axes[2]

latest_year  = trade['year'].max()
trade_latest = trade[
    trade['year'] == latest_year
].sort_values('qty_kg', ascending=False)

bar_colors = [colors[c] for c in trade_latest['country']]
bars = ax3.bar(
    trade_latest['country'],
    trade_latest['qty_kg'] / 1e6,
    color=bar_colors,
    alpha=0.85
)

for bar, val in zip(bars, trade_latest['qty_kg']):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{val/1e6:.1f}M kg",
        ha='center', fontsize=9, fontweight='bold'
    )

cn = trade_latest[
    trade_latest['country'] == 'China'
]['qty_kg'].values[0]
au = trade_latest[
    trade_latest['country'] == 'Australia'
]['qty_kg'].values[0]

ax3.set_title(f'{latest_year} Export Volume Comparison')
ax3.set_xlabel('Country')
ax3.set_ylabel('Export Volume (million kg)')
ax3.text(
    0.5, 0.95,
    f'China exports {int(cn/au)}x more than Australia',
    transform=ax3.transAxes,
    ha='center', fontsize=9,
    color='#F44336', fontweight='bold'
)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()

# ── Save to student's Google Drive ───────────────────────────
save_path = OUTPUT_FOLDER + 'chart_trade_flows.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved to: {save_path}")

---
## Section 5 — Interactive Supply Chain Map

Plotting all 15 supply chain nodes on an interactive world map using coordinates loaded from source files. Mining sites, processing plants, and manufacturers are shown with different colors. Trade routes connect each supply chain path.

In [ ]:
# ============================================================
# SECTION 5 — INTERACTIVE SUPPLY CHAIN MAP
# Using Plotly — renders inline in Google Colab
# Data source: master table from Section 3
# ============================================================

import plotly.graph_objects as go

# Color per tier — visual styling only
tier_colors = {
    'Mining':        'red',
    'Processing':    'orange',
    'Manufacturing': 'blue'
}

# Color per path — visual styling only
path_colors = {
    1: '#E53935',
    2: '#43A047',
    3: '#1E88E5',
    4: '#FB8C00',
    5: '#8E24AA'
}

fig = go.Figure()

# --------------------------------------------------------
# Add supply chain path lines
# Built from master table
# --------------------------------------------------------
tier_order = {'Mining': 0, 'Processing': 1,
              'Manufacturing': 2}

for path_id in sorted(master['path_id'].unique()):
    path_nodes = master[
        master['path_id'] == path_id
    ].copy()
    path_nodes['order'] = path_nodes['tier'].map(
        tier_order)
    path_nodes = path_nodes.sort_values('order')
    path_nodes = path_nodes.dropna(
        subset=['latitude','longitude'])

    fig.add_trace(go.Scattergeo(
        lon=path_nodes['longitude'].tolist(),
        lat=path_nodes['latitude'].tolist(),
        mode='lines',
        line=dict(
            width=2,
            color=path_colors[path_id]
        ),
        name=f'Path {path_id}',
        showlegend=True
    ))

# --------------------------------------------------------
# Add nodes per tier
# Built from master table
# --------------------------------------------------------
for tier in ['Mining', 'Processing', 'Manufacturing']:
    tier_nodes = master[
        master['tier'] == tier
    ].dropna(subset=['latitude','longitude'])

    fig.add_trace(go.Scattergeo(
        lon=tier_nodes['longitude'].tolist(),
        lat=tier_nodes['latitude'].tolist(),
        mode='markers',
        marker=dict(
            size=12,
            color=tier_colors[tier],
            symbol='circle',
            line=dict(width=1, color='white')
        ),
        name=tier,
        text=tier_nodes.apply(
            lambda r:
            f"<b>{r['node_name']}</b><br>"
            f"Company: {r['company']}<br>"
            f"Country: {r['country']}<br>"
            f"Path: {r['path_id']}<br>"
            f"Source: {r['source_document']}<br>"
            f"Coord source: {r['coord_source']}",
            axis=1
        ).tolist(),
        hoverinfo='text',
        showlegend=True
    ))

# Layout
fig.update_layout(
    title=dict(
        text='Global Lithium Supply Chain — '
             'Plausibility Assessment<br>'
             '<sub>Sources: GA Critical Minerals 2025 | '
             'USGS MRDS | OpenStreetMap Nominatim</sub>',
        x=0.5
    ),
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor='lightgray',
        showland=True,
        landcolor='#f5f5f5',
        showocean=True,
        oceancolor='#e8f4f8',
        showlakes=True,
        lakecolor='#e8f4f8',
        projection_type='natural earth'
    ),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='lightgray',
        borderwidth=1
    ),
    height=600,
    margin=dict(l=0, r=0, t=80, b=0)
)

# Save to Google Drive
map_path = OUTPUT_FOLDER + 'supply_chain_map.html'
fig.write_html(map_path)

print(f"Map saved to: {map_path}")
print(f"Nodes plotted: {len(master)}")
print(f"Paths drawn: {master['path_id'].nunique()}")

fig.show()

---
## Section 6 — Knowledge Graph

Representing the lithium supply chain as a knowledge graph using NetworkX and Plotly. Built entirely from the master table and trade data loaded in Section 3.

**Node types:**
- Red circles → Mining facilities
- Orange circles → Processing facilities
- Blue circles → Manufacturing facilities
- Purple squares → Companies (hover for details)
- Green diamonds → Countries

**Edge types:**
- White solid → Material flow
- Purple dotted → Ownership
- Teal dashed → Location
- Orange solid → Trade flow

In [ ]:
# ============================================================
# SECTION 6 — KNOWLEDGE GRAPH
# Built from master table and trade data
# No hardcoded values
# ============================================================

import networkx as nx
import plotly.graph_objects as go

# ------------------------------------------------------------
# Build graph
# ------------------------------------------------------------

# ── Ensure trade variable is available ───────────────────────
if 'trade' not in dir():
    trade = comtrade.copy()
    trade['price_per_kg'] = (
        trade['fob_usd'] / trade['qty_kg']
    ).round(2)

G = nx.DiGraph()

# Facility nodes from master table
for _, row in master.iterrows():
    G.add_node(
        row['node_name'],
        node_type='Facility',
        tier=row['tier'],
        company=row['company'],
        country=row['country'],
        source=row['source_document'],
        coord_source=row['coord_source']
    )

# Company nodes from master table
for company in master['company'].unique():
    if company not in G.nodes:
        G.add_node(
            company,
            node_type='Company',
            tier='Owner'
        )

# Country nodes from master table + trade data
for country in master['country'].unique():
    if country not in G.nodes:
        ct = trade[trade['country'] == country]
        if len(ct) > 0:
            latest = ct[
                ct['year'] == ct['year'].max()
            ].iloc[0]
            G.add_node(
                country,
                node_type='Country',
                tier='Region',
                export_kg=float(latest['qty_kg']),
                price_per_kg=float(
                    latest['price_per_kg']),
                year=int(latest['year'])
            )
        else:
            G.add_node(
                country,
                node_type='Country',
                tier='Region'
            )

# ------------------------------------------------------------
# Add edges from data
# ------------------------------------------------------------

tier_order = {
    'Mining': 0,
    'Processing': 1,
    'Manufacturing': 2
}

# Material flow from master table
for path_id in master['path_id'].unique():
    path_nodes = master[
        master['path_id'] == path_id
    ].copy()
    path_nodes['order'] = path_nodes['tier'].map(
        tier_order)
    path_nodes = path_nodes.sort_values('order')
    nodes_list = path_nodes['node_name'].tolist()
    for i in range(len(nodes_list) - 1):
        G.add_edge(
            nodes_list[i],
            nodes_list[i+1],
            edge_type='Material Flow',
            path_id=int(path_id)
        )

# Ownership from master table
for _, row in master.iterrows():
    G.add_edge(
        row['company'],
        row['node_name'],
        edge_type='Ownership',
        source=row['source_document']
    )

# Location from master table
for _, row in master.iterrows():
    G.add_edge(
        row['node_name'],
        row['country'],
        edge_type='Located In'
    )

# Trade flows from Comtrade data
trade_latest = trade[
    trade['year'] == trade['year'].max()
]
trade_routes = [
    ('Australia', 'China'),
    ('Chile', 'South Korea'),
    ('China', 'Japan'),
    ('China', 'South Korea'),
]
for src, dst in trade_routes:
    vol = trade_latest[
        trade_latest['country'] == src
    ]['qty_kg'].sum()
    G.add_edge(
        src, dst,
        edge_type='Trade Flow',
        volume_kg=float(vol),
        source='UN Comtrade 2024'
    )

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("=" * 60)
print("KNOWLEDGE GRAPH STATISTICS")
print("=" * 60)
print(f"Total nodes: {G.number_of_nodes()}")
print(f"Total edges: {G.number_of_edges()}")

node_types = {}
for _, d in G.nodes(data=True):
    t = d.get('node_type', 'Unknown')
    node_types[t] = node_types.get(t, 0) + 1
print("\nNode breakdown:")
for t, c in node_types.items():
    print(f"  {t}: {c}")

edge_types = {}
for _, _, d in G.edges(data=True):
    t = d.get('edge_type', 'Unknown')
    edge_types[t] = edge_types.get(t, 0) + 1
print("\nEdge breakdown:")
for t, c in edge_types.items():
    print(f"  {t}: {c}")

centrality  = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G)

print("\nTop 5 most connected nodes:")
for node, score in sorted(
    centrality.items(),
    key=lambda x: x[1], reverse=True
)[:5]:
    ntype = G.nodes[node].get('node_type', '')
    print(f"  {node} ({ntype}): {score:.3f}")

print("\nTop 5 betweenness centrality:")
for node, score in sorted(
    betweenness.items(),
    key=lambda x: x[1], reverse=True
)[:5]:
    ntype = G.nodes[node].get('node_type', '')
    print(f"  {node} ({ntype}): {score:.3f}")

# ------------------------------------------------------------
# Node positions — 5 clear zones
# ------------------------------------------------------------

facility_nodes = [
    n for n, d in G.nodes(data=True)
    if d.get('node_type') == 'Facility'
]
company_nodes = [
    n for n, d in G.nodes(data=True)
    if d.get('node_type') == 'Company'
]
country_nodes = [
    n for n, d in G.nodes(data=True)
    if d.get('node_type') == 'Country'
]

pos = {}

# Facilities — center 3 columns
tier_x = {
    'Mining': 2,
    'Processing': 5,
    'Manufacturing': 8
}
tier_nodes_dict = {
    'Mining': [],
    'Processing': [],
    'Manufacturing': []
}
for n in facility_nodes:
    t = G.nodes[n].get('tier', 'Mining')
    tier_nodes_dict[t].append(n)

for tier, nodes_list in tier_nodes_dict.items():
    n = len(nodes_list)
    for i, node in enumerate(nodes_list):
        y = (i - (n-1)/2) * 2.5
        pos[node] = (tier_x[tier], y)

# Companies — far left
n = len(company_nodes)
for i, node in enumerate(company_nodes):
    y = (i - (n-1)/2) * 1.8
    pos[node] = (-2, y)

# Countries — far right
n = len(country_nodes)
for i, node in enumerate(country_nodes):
    y = (i - (n-1)/2) * 2.2
    pos[node] = (12, y)

# ------------------------------------------------------------
# Plotly visualization
# ------------------------------------------------------------

edge_styles = {
    'Material Flow': {
        'color': '#FFFFFF',
        'dash': 'solid',
        'width': 2.5
    },
    'Ownership': {
        'color': '#CE93D8',
        'dash': 'dot',
        'width': 1.2
    },
    'Located In': {
        'color': '#80CBC4',
        'dash': 'dash',
        'width': 1
    },
    'Trade Flow': {
        'color': '#FFB74D',
        'dash': 'solid',
        'width': 2.5
    }
}

tier_colors = {
    'Mining':        '#E53935',
    'Processing':    '#FB8C00',
    'Manufacturing': '#1E88E5',
    'Owner':         '#CE93D8',
    'Region':        '#81C784'
}

fig = go.Figure()

# Draw edges — one trace per type
legend_shown = set()
for edge_type, style in edge_styles.items():
    x_vals, y_vals = [], []
    for src, dst, attrs in G.edges(data=True):
        if attrs.get('edge_type') != edge_type:
            continue
        x0, y0 = pos[src]
        x1, y1 = pos[dst]
        x_vals += [x0, x1, None]
        y_vals += [y0, y1, None]

    if x_vals:
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=y_vals,
            mode='lines',
            line=dict(
                color=style['color'],
                dash=style['dash'],
                width=style['width']
            ),
            hoverinfo='skip',
            name=edge_type,
            showlegend=edge_type not in legend_shown,
            legendgroup=edge_type
        ))
        legend_shown.add(edge_type)

# Draw nodes — one legend entry per category
node_legend_shown = set()

all_nodes = (
    [('Facility', n) for n in facility_nodes] +
    [('Company', n) for n in company_nodes] +
    [('Country', n) for n in country_nodes]
)

for node_type, node in all_nodes:
    attrs = G.nodes[node]
    x, y  = pos[node]
    tier  = attrs.get('tier', '')
    color = tier_colors.get(tier, '#FFFFFF')
    size  = 18 + centrality.get(node, 0) * 60

    # Hover text
    hover = f"<b>{node}</b>"
    hover += f"<br>Type: {node_type}"
    if tier and tier not in ['Owner', 'Region']:
        hover += f"<br>Tier: {tier}"
    if 'company' in attrs and node_type == 'Facility':
        hover += f"<br>Owner: {attrs['company']}"
    if 'country' in attrs and node_type == 'Facility':
        hover += f"<br>Country: {attrs['country']}"
    if 'source' in attrs:
        hover += f"<br>Source: {attrs['source']}"
    if 'coord_source' in attrs:
        hover += f"<br>Coord: {attrs['coord_source']}"
    if 'export_kg' in attrs:
        hover += (
            f"<br>Export {attrs.get('year', '')}: "
            f"{attrs['export_kg']/1e6:.1f}M kg"
            f"<br>Price: ${attrs['price_per_kg']}/kg"
        )

    legend_name = (
        f"Facility ({tier})"
        if node_type == 'Facility'
        else node_type
    )
    show_leg = legend_name not in node_legend_shown
    node_legend_shown.add(legend_name)

    # Text position per node type
    if node_type == 'Company':
        text_pos = 'middle left'
    elif node_type == 'Country':
        text_pos = 'middle right'
    else:
        text_pos = 'top center'

    fig.add_trace(go.Scatter(
        x=[x], y=[y],
        mode='markers+text',
        marker=dict(
            size=size,
            color=color,
            symbol=(
                'diamond' if node_type == 'Country'
                else 'square' if node_type == 'Company'
                else 'circle'
            ),
            line=dict(width=1.5, color='white')
        ),
        text=node,
        textposition=text_pos,
        textfont=dict(size=8, color='white'),
        hovertext=hover,
        hoverinfo='text',
        name=legend_name,
        showlegend=show_leg,
        legendgroup=legend_name
    ))

# Zone labels
for label, x, color in [
    ('Companies', -2, '#CE93D8'),
    ('Mining', 2, '#E53935'),
    ('Processing', 5, '#FB8C00'),
    ('Manufacturing', 8, '#1E88E5'),
    ('Countries', 12, '#81C784')
]:
    fig.add_annotation(
        x=x, y=8.5,
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(size=12, color=color)
    )

fig.update_layout(
    title=dict(
        text=(
            'Lithium Supply Chain — Knowledge Graph<br>'
            '<sub>'
            '◆ Country  ■ Company  ● Facility  |  '
            'White = Material Flow  |  '
            'Purple = Ownership  |  '
            'Teal = Location  |  '
            'Orange = Trade Flow'
            '</sub>'
        ),
        x=0.5,
        font=dict(size=13, color='white')
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        range=[-5, 15]
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        range=[-8, 10]
    ),
    height=750,
    showlegend=True,
    legend=dict(
        bgcolor='rgba(40,40,60,0.9)',
        font=dict(color='white', size=10),
        bordercolor='gray',
        borderwidth=1,
        x=1.01, y=1
    ),
    hoverlabel=dict(
        bgcolor='#2a2a3e',
        font=dict(color='white', size=11)
    ),
    margin=dict(l=20, r=160, t=100, b=20)
)

# Save
kg_path = OUTPUT_FOLDER + 'knowledge_graph.html'
fig.write_html(kg_path)
print(f"Knowledge graph saved to: {kg_path}")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

fig.show()

---
## Section 7 — Plausibility Assessment

Running three plausibility rules against each supply chain path using data loaded from source files.

**Rule 1 — Geological Consistency**
Does the geology at the declared mine location actually support lithium extraction?

**Rule 2 — Geographic Feasibility**
Is the declared trade route between nodes physically and logistically realistic?

**Rule 3 — Network Structure Completeness**
Is every tier present — Mining, Processing, Manufacturing?

**Scoring:**
- Each rule scores 1.0 (pass) or 0.0 (fail)
- Overall score = average of three rules
- Score ≥ 0.8 → PASS
- Score ≥ 0.5 → WARN
- Score < 0.5 → FAIL

In [ ]:
# ============================================================
# SECTION 7 — PLAUSIBILITY ASSESSMENT
# Three rules applied to all 5 supply chain paths
# All checks use data loaded from source files
# ============================================================

import math
import plotly.graph_objects as go

# ------------------------------------------------------------
# RULE 1 — Geological Consistency
# Source: USGS MRDS records loaded in Section 3
# Check: Does MRDS confirm lithium at declared mine location?
# ------------------------------------------------------------

print("=" * 60)
print("RULE 1 — Geological Consistency")
print("Source: USGS MRDS + GA Critical Minerals 2025")
print("=" * 60)

# Extract lithium confirmed mines from MRDS
def check_geology(mine_name, path_id):
    """
    Check if MRDS confirms lithium at mine location
    Returns score and explanation
    """
    # Check GA dataset for Australian mines
    ga_match = ga_li[ga_li['Name'] == mine_name]
    if len(ga_match) > 0:
        commodity = ga_match['Commodity Group'].values[0]
        status = ga_match['Status'].values[0]
        if 'Lithium' in commodity:
            return 1.0, f"GA 2025 confirms: {commodity} | {status}"
        else:
            return 0.0, f"GA 2025: Lithium NOT confirmed"

    # Check MRDS for non-Australian mines
    for name, df in mrds.items():
        if len(df) == 0:
            continue
        if 'commod1' not in df.columns:
            continue
        li_rows = df[df['commod1'].str.contains(
            'Lithium', na=False, case=False)]
        if len(li_rows) > 0:
            row = li_rows.iloc[0]
            site = str(row.get('site_name', ''))
            status = str(row.get('dev_stat', ''))
            return 1.0, f"MRDS confirms Lithium | {status}"

    return 0.5, "Lithium confirmed from annual reports only"

# Apply to all 5 mining nodes
mining_nodes = master[master['tier'] == 'Mining']
rule1_scores = {}

for _, row in mining_nodes.iterrows():
    score, explanation = check_geology(
        row['node_name'], row['path_id'])
    rule1_scores[row['path_id']] = {
        'score': score,
        'explanation': explanation,
        'mine': row['node_name'],
        'source': row['source_document']
    }
    print(f"\nPath {row['path_id']}: {row['node_name']}")
    print(f"  Score: {score}")
    print(f"  {explanation}")

# ------------------------------------------------------------
# RULE 2 — Geographic Feasibility
# Source: Coordinates from GA, MRDS, OSM Nominatim
# Check: Is shipping distance realistic?
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("RULE 2 — Geographic Feasibility")
print("Source: Coordinates from GA + MRDS + OSM Nominatim")
print("=" * 60)

def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate distance in km between two coordinates
    Using Haversine formula
    """
    R = 6371  # Earth radius km
    lat1, lon1, lat2, lon2 = map(
        math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (math.sin(dlat/2)**2 +
         math.cos(lat1) * math.cos(lat2) *
         math.sin(dlon/2)**2)
    return R * 2 * math.asin(math.sqrt(a))

# Maximum feasible distances per route type
MAX_MINE_TO_PROCESSOR_KM = 5000
MAX_PROCESSOR_TO_MFG_KM  = 15000

rule2_scores = {}

for path_id in master['path_id'].unique():
    path_nodes = master[
        master['path_id'] == path_id
    ].copy()
    path_nodes['order'] = path_nodes['tier'].map(
        {'Mining': 0, 'Processing': 1, 'Manufacturing': 2})
    path_nodes = path_nodes.sort_values('order')

    scores = []
    details = []

    for i in range(len(path_nodes) - 1):
        n1 = path_nodes.iloc[i]
        n2 = path_nodes.iloc[i+1]

        if pd.isna(n1['latitude']) or pd.isna(n2['latitude']):
            scores.append(0.5)
            details.append("Coordinates unavailable")
            continue

        dist = haversine(
            n1['latitude'], n1['longitude'],
            n2['latitude'], n2['longitude']
        )

        threshold = (
            MAX_MINE_TO_PROCESSOR_KM
            if i == 0
            else MAX_PROCESSOR_TO_MFG_KM
        )

        if dist <= threshold:
            score = 1.0
            verdict = "FEASIBLE"
        else:
            score = 0.5
            verdict = "EXCEEDS THRESHOLD — flagged for review"

        scores.append(score)
        details.append(
            f"{n1['node_name']} → {n2['node_name']}: "
            f"{dist:.0f} km ({verdict})"
        )

    avg_score = sum(scores) / len(scores)
    rule2_scores[path_id] = {
        'score': avg_score,
        'details': details
    }

    print(f"\nPath {path_id}: Score = {avg_score}")
    for d in details:
        print(f"  {d}")

# ------------------------------------------------------------
# RULE 3 — Network Structure Completeness
# Source: master table built from annual reports
# Check: Are all 3 tiers present?
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("RULE 3 — Network Structure Completeness")
print("Source: Master table from annual reports")
print("=" * 60)

required_tiers = {'Mining', 'Processing', 'Manufacturing'}
rule3_scores = {}

for path_id in master['path_id'].unique():
    path_nodes = master[
        master['path_id'] == path_id
    ]
    present_tiers = set(path_nodes['tier'].tolist())
    missing = required_tiers - present_tiers
    score = len(present_tiers) / len(required_tiers)

    rule3_scores[path_id] = {
        'score': score,
        'present': present_tiers,
        'missing': missing
    }

    print(f"\nPath {path_id}: Score = {score}")
    print(f"  Present tiers: {present_tiers}")
    if missing:
        print(f"  Missing tiers: {missing}")
    else:
        print(f"  All tiers present ✅")

# ------------------------------------------------------------
# FINAL PLAUSIBILITY REPORT
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("FINAL PLAUSIBILITY REPORT")
print("=" * 60)

results = []
path_names = {
    1: 'Greenbushes → Kwinana → CATL',
    2: 'Atacama → La Negra → LG Chem',
    3: 'Wodgina → Kemerton → Panasonic',
    4: 'Qinghai → Ganfeng → BYD',
    5: 'Pilgangoora → China → Samsung SDI'
}

for path_id in sorted(master['path_id'].unique()):
    r1 = rule1_scores[path_id]['score']
    r2 = rule2_scores[path_id]['score']
    r3 = rule3_scores[path_id]['score']
    overall = (r1 + r2 + r3) / 3

    if overall >= 0.8:
        verdict = 'PASS'
    elif overall >= 0.5:
        verdict = 'WARN'
    else:
        verdict = 'FAIL'

    results.append({
        'path_id': path_id,
        'path': path_names[path_id],
        'rule1_geology': r1,
        'rule2_distance': r2,
        'rule3_structure': r3,
        'overall': round(overall, 2),
        'verdict': verdict
    })

results_df = pd.DataFrame(results)
print(results_df[[
    'path_id', 'path',
    'rule1_geology', 'rule2_distance',
    'rule3_structure', 'overall', 'verdict'
]].to_string(index=False))

# ------------------------------------------------------------
# VISUALIZE RESULTS
# ------------------------------------------------------------

verdict_colors = {
    'PASS': '#4CAF50',
    'WARN': '#FF9800',
    'FAIL': '#F44336'
}

fig = go.Figure()

rules = [
    ('rule1_geology', 'Rule 1: Geology'),
    ('rule2_distance', 'Rule 2: Distance'),
    ('rule3_structure', 'Rule 3: Structure')
]

rule_colors = ['#2196F3', '#9C27B0', '#FF5722']

for (col, label), color in zip(rules, rule_colors):
    fig.add_trace(go.Bar(
        name=label,
        x=results_df['path'].tolist(),
        y=results_df[col].tolist(),
        marker_color=color,
        opacity=0.85
    ))

# Add overall score line
fig.add_trace(go.Scatter(
    name='Overall Score',
    x=results_df['path'].tolist(),
    y=results_df['overall'].tolist(),
    mode='lines+markers+text',
    line=dict(color='white', width=2),
    marker=dict(size=10, color='white'),
    text=[
        f"{v}<br>{s}"
        for v, s in zip(
            results_df['verdict'],
            results_df['overall']
        )
    ],
    textposition='top center',
    textfont=dict(size=10, color='white')
))

# Threshold lines
fig.add_hline(
    y=0.8, line_dash='dash',
    line_color='#4CAF50',
    annotation_text='PASS threshold (0.8)',
    annotation_position='right'
)
fig.add_hline(
    y=0.5, line_dash='dash',
    line_color='#FF9800',
    annotation_text='WARN threshold (0.5)',
    annotation_position='right'
)

fig.update_layout(
    title=dict(
        text='Plausibility Assessment Results — All 5 Paths<br>'
             '<sub>Source: USGS MRDS | GA Critical Minerals 2025 | '
             'OSM Nominatim | Annual Reports</sub>',
        x=0.5,
        font=dict(color='white', size=13)
    ),
    barmode='group',
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    xaxis=dict(
        tickangle=-15,
        gridcolor='#2a2a3e'
    ),
    yaxis=dict(
        range=[0, 1.2],
        title='Score',
        gridcolor='#2a2a3e'
    ),
    legend=dict(
        bgcolor='rgba(40,40,60,0.9)',
        font=dict(color='white')
    ),
    height=550,
    margin=dict(l=40, r=40, t=100, b=80)
)

# Save
plaus_path = OUTPUT_FOLDER + 'plausibility_results.html'
fig.write_html(plaus_path)
print(f"\nPlausibility chart saved to: {plaus_path}")

fig.show()

---
## Section 8 — Remote Sensing

Using Google Earth Engine to access Sentinel-2 satellite imagery over all 5 mine locations. We calculate NDVI (Normalized Difference Vegetation Index) to detect active mining footprints — active mines show low NDVI due to bare rock and disturbed land.

**Mine locations analysed:**
- Greenbushes, Western Australia
- Wodgina, Western Australia
- Pilgangoora, Western Australia
- Salar de Atacama, Chile
- Qinghai Lake Brine, China

**What NDVI tells us:**
- NDVI close to 0 or negative → bare rock, active mining
- NDVI 0.2 to 0.5 → sparse vegetation
- NDVI above 0.6 → dense vegetation, no mining activity

In [ ]:
# ============================================================
# SECTION 8 — STEP 1
# Authenticate and Initialize Google Earth Engine
# ============================================================
#
# >>> ONE-TIME SETUP (only needed once per Google account) <
#
# If this is your first time:
#
#   1. Go to: https://code.earthengine.google.com
#   2. Sign in with the SAME Google account you used
#      for Google Drive in Section 2
#   3. Accept the terms of service
#      (non-commercial use is approved instantly)
#   4. Earth Engine automatically creates a Cloud Project
#      for you — usually named something like "ee-yourname"
#   5. Find your Project ID at:
#      https://console.cloud.google.com
#      → top-left dropdown → copy the "ID" (not the Name)
#   6. Paste that ID into MY_GEE_PROJECT below
#
# Then run this cell. A browser popup will ask you to
# sign in — use the SAME account as your Drive account.
# ============================================================

import ee

# ── EDIT THIS LINE WITH YOUR OWN PROJECT ID ──────────────────
MY_GEE_PROJECT = 'lithium-plausibility'
# Example: MY_GEE_PROJECT = 'ee-aliexample'

print("=" * 60)
print("SECTION 8 — Google Earth Engine Setup")
print("=" * 60)
print()

if MY_GEE_PROJECT == 'PASTE_YOUR_PROJECT_ID_HERE':
    print("STOP — You need to set your Project ID first.")
    print()
    print("Follow the steps in the comment block above this")
    print("cell, then replace PASTE_YOUR_PROJECT_ID_HERE")
    print("with your actual Earth Engine project ID.")
    print()
    print("This is a ONE-TIME setup — takes about 2 minutes.")

else:
    print("Authenticating...")
    print("IMPORTANT: Use the SAME Google account you used")
    print("for Google Drive in Section 2.")
    print()

    try:
        ee.Authenticate()
        ee.Initialize(project=MY_GEE_PROJECT)
        print()
        print(f"SUCCESS — Earth Engine initialized")
        print(f"Project: {MY_GEE_PROJECT}")
        print()
        print("You can now run Steps 2-5 below.")

    except Exception as e:
        print()
        print("=" * 60)
        print("INITIALIZATION FAILED")
        print("=" * 60)
        print()
        print("Check these common issues:")
        print()
        print("1. Project ID is wrong")
        print("   → Use the PROJECT ID (e.g. ee-yourname-12345)")
        print("     not the Project NAME or display name")
        print("   → Find it at console.cloud.google.com")
        print()
        print("2. You signed in with a different account")
        print("   → Must match the account registered at")
        print("     code.earthengine.google.com")
        print()
        print("3. You have not completed EE signup yet")
        print("   → Visit code.earthengine.google.com")
        print("     and accept the terms first")
        print()
        print(f"Error detail: {e}")

In [ ]:
# ============================================================
# STEP 2 — Extract mine coordinates from master table
# Using coordinates already loaded from source files
# ============================================================

print("=" * 60)
print("STEP 2 — Mine Locations from Master Table")
print("=" * 60)

# Get mining nodes from master table
mining_nodes = master[
    master['tier'] == 'Mining'
].copy()

print("Mine locations for remote sensing analysis:")
for _, row in mining_nodes.iterrows():
    print(f"\nPath {row['path_id']}: {row['node_name']}")
    print(f"  Lat: {row['latitude']}")
    print(f"  Lon: {row['longitude']}")
    print(f"  Coord source: {row['coord_source']}")

# Build mine locations dict from master table
mine_locations = {}
for _, row in mining_nodes.iterrows():
    mine_locations[row['node_name']] = {
        'lat': row['latitude'],
        'lon': row['longitude'],
        'path_id': row['path_id'],
        'source': row['coord_source']
    }

print(f"\nTotal mines for analysis: {len(mine_locations)}")

In [ ]:
# ============================================================
# STEP 3 — Sentinel-2 NDVI Analysis
# For each mine location we:
# 1. Create a buffer zone around the coordinates
# 2. Load Sentinel-2 imagery (cloud filtered)
# 3. Calculate NDVI
# 4. Extract mean NDVI value
# 5. Interpret result for plausibility check
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print("=" * 60)
print("STEP 3 — Sentinel-2 NDVI Analysis")
print("Source: Copernicus Sentinel-2 via Google Earth Engine")
print("Period: 2023-01-01 to 2024-12-31")
print("Cloud cover filter: less than 20 percent")
print("=" * 60)

# NDVI interpretation thresholds
# Based on standard remote sensing practice
NDVI_ACTIVE_MINING  = 0.15
NDVI_SPARSE_VEG     = 0.35

def analyse_mine_ndvi(mine_name, lat, lon):
    """
    Loads Sentinel-2 imagery over mine location
    Calculates mean NDVI
    Returns NDVI value and plausibility interpretation
    """
    try:
        # Create point and buffer from coordinates
        point = ee.Geometry.Point([lon, lat])
        area  = point.buffer(5000)  # 5km buffer

        # Load Sentinel-2 Surface Reflectance
        # Filter by date, location, and cloud cover
        s2 = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(area)
            .filterDate('2023-01-01', '2024-12-31')
            .filter(
                ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)
            )
            .median()
        )

        # Calculate NDVI
        # NDVI = (NIR - Red) / (NIR + Red)
        # Sentinel-2: NIR = B8, Red = B4
        ndvi = s2.normalizedDifference(['B8', 'B4'])

        # Extract mean NDVI over mine area
        stats = ndvi.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=area,
            scale=10,
            maxPixels=1e9
        )

        ndvi_val = stats.getInfo().get('nd', None)

        if ndvi_val is None:
            return None, 'No imagery available'

        ndvi_val = round(ndvi_val, 4)

        # Interpret NDVI for plausibility
        if ndvi_val < NDVI_ACTIVE_MINING:
            interpretation = (
                'Active mining confirmed — '
                'bare rock and disturbed land detected'
            )
            plausibility = 'CONFIRMED'
        elif ndvi_val < NDVI_SPARSE_VEG:
            interpretation = (
                'Sparse vegetation — '
                'consistent with active or recent mining'
            )
            plausibility = 'CONSISTENT'
        else:
            interpretation = (
                'Dense vegetation — '
                'mining activity not clearly visible'
            )
            plausibility = 'UNCERTAIN'

        return ndvi_val, interpretation, plausibility

    except Exception as e:
        return None, str(e), 'ERROR'

# Run NDVI analysis for all 5 mines
ndvi_results = {}

for mine_name, info in mine_locations.items():
    print(f"\nAnalysing: {mine_name}")
    print(f"  Coordinates: {info['lat']}, {info['lon']}")
    print(f"  Source: {info['source']}")

    result = analyse_mine_ndvi(
        mine_name,
        info['lat'],
        info['lon']
    )

    if len(result) == 3:
        ndvi_val, interpretation, plausibility = result
    else:
        ndvi_val, interpretation = result
        plausibility = 'ERROR'

    ndvi_results[mine_name] = {
        'ndvi': ndvi_val,
        'interpretation': interpretation,
        'plausibility': plausibility,
        'path_id': info['path_id'],
        'lat': info['lat'],
        'lon': info['lon']
    }

    print(f"  NDVI: {ndvi_val}")
    print(f"  Interpretation: {interpretation}")
    print(f"  Plausibility: {plausibility}")

print("\n")
print("=" * 60)
print("NDVI Analysis Complete")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 4 — Visualize NDVI Results
# Charts + satellite map for each mine
# ============================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------------------------
# CHART 1 — NDVI comparison across all mines
# ------------------------------------------------------------

mine_names  = list(ndvi_results.keys())
ndvi_values = [ndvi_results[m]['ndvi'] for m in mine_names]
plaus_vals  = [ndvi_results[m]['plausibility']
               for m in mine_names]

# Color per plausibility verdict
bar_colors = []
for p in plaus_vals:
    if p == 'CONFIRMED':
        bar_colors.append('#4CAF50')
    elif p == 'CONSISTENT':
        bar_colors.append('#FF9800')
    elif p == 'UNCERTAIN':
        bar_colors.append('#F44336')
    else:
        bar_colors.append('#9E9E9E')

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=mine_names,
    y=ndvi_values,
    marker_color=bar_colors,
    text=[
        f"NDVI: {v:.4f}<br>{p}"
        for v, p in zip(ndvi_values, plaus_vals)
    ],
    textposition='outside',
    textfont=dict(size=9, color='white'),
    hovertext=[
        f"<b>{m}</b><br>"
        f"NDVI: {ndvi_results[m]['ndvi']}<br>"
        f"Status: {ndvi_results[m]['plausibility']}<br>"
        f"Path: {ndvi_results[m]['path_id']}<br>"
        f"{ndvi_results[m]['interpretation']}"
        for m in mine_names
    ],
    hoverinfo='text'
))

# Add threshold lines
fig1.add_hline(
    y=NDVI_ACTIVE_MINING,
    line_dash='dash',
    line_color='#4CAF50',
    annotation_text='Active mining threshold (0.15)',
    annotation_position='right',
    annotation_font_color='#4CAF50'
)
fig1.add_hline(
    y=NDVI_SPARSE_VEG,
    line_dash='dash',
    line_color='#FF9800',
    annotation_text='Sparse vegetation threshold (0.35)',
    annotation_position='right',
    annotation_font_color='#FF9800'
)

# Add legend patches
for label, color in [
    ('CONFIRMED — Active mining', '#4CAF50'),
    ('CONSISTENT — Sparse vegetation', '#FF9800'),
    ('UNCERTAIN — Dense vegetation', '#F44336')
]:
    fig1.add_trace(go.Bar(
        x=[None], y=[None],
        marker_color=color,
        name=label,
        showlegend=True
    ))

fig1.update_layout(
    title=dict(
        text='Sentinel-2 NDVI Analysis — All 5 Mine Sites<br>'
             '<sub>Source: Copernicus Sentinel-2 via GEE | '
             'Period: 2023-2024 | Cloud cover < 20%</sub>',
        x=0.5,
        font=dict(color='white', size=13)
    ),
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    xaxis=dict(
        tickangle=-15,
        gridcolor='#2a2a3e'
    ),
    yaxis=dict(
        title='NDVI Value',
        range=[-0.1, 0.8],
        gridcolor='#2a2a3e'
    ),
    legend=dict(
        bgcolor='rgba(40,40,60,0.9)',
        font=dict(color='white', size=10)
    ),
    height=550,
    margin=dict(l=40, r=200, t=100, b=80)
)

# Save chart
ndvi_chart_path = OUTPUT_FOLDER + 'ndvi_analysis.html'
fig1.write_html(ndvi_chart_path)
fig1.show()

print(f"NDVI chart saved to: {ndvi_chart_path}")

# ------------------------------------------------------------
# MAP — Mine locations with NDVI values
# ------------------------------------------------------------

import folium

m = folium.Map(
    location=[10, 100],
    zoom_start=2,
    tiles='CartoDB dark_matter'
)

# Color per plausibility
color_map = {
    'CONFIRMED':  'green',
    'CONSISTENT': 'orange',
    'UNCERTAIN':  'red',
    'ERROR':      'gray'
}

for mine_name, info in ndvi_results.items():
    color = color_map.get(info['plausibility'], 'gray')
    ndvi  = info['ndvi']

    popup_text = f"""
    <b>{mine_name}</b><br>
    NDVI: {ndvi}<br>
    Status: {info['plausibility']}<br>
    {info['interpretation']}<br>
    Path: {info['path_id']}
    """

    folium.CircleMarker(
        location=[info['lat'], info['lon']],
        radius=15,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=250),
        tooltip=f"{mine_name} | NDVI: {ndvi}"
    ).add_to(m)

    # Add NDVI label
    folium.Marker(
        location=[info['lat'], info['lon']],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size:10px;
                color:white;
                font-weight:bold;
                text-shadow: 1px 1px 2px black;
                white-space:nowrap;
                margin-top:-30px;
                margin-left:15px
            ">
            {mine_name.split()[0]}<br>
            NDVI:{ndvi}
            </div>
            """,
            icon_size=(120, 40)
        )
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position:fixed;bottom:30px;left:30px;
     z-index:1000;background:#1a1a2e;padding:12px;
     border-radius:8px;border:1px solid #ccc;
     font-family:Arial;font-size:12px;color:white">
<b>NDVI Plausibility</b><br><br>
<span style="color:#4CAF50">●</span>
CONFIRMED (NDVI < 0.15)<br>
<span style="color:#FF9800">●</span>
CONSISTENT (NDVI 0.15-0.35)<br>
<span style="color:#F44336">●</span>
UNCERTAIN (NDVI > 0.35)<br><br>
<b>Source:</b> Sentinel-2 via GEE<br>
<b>Period:</b> 2023-2024
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Save map
ndvi_map_path = OUTPUT_FOLDER + 'ndvi_map.html'
m.save(ndvi_map_path)

print(f"\nNDVI map saved to: {ndvi_map_path}")
print("\nKey findings:")
for mine_name, info in ndvi_results.items():
    print(
        f"  {mine_name}: "
        f"NDVI={info['ndvi']} → {info['plausibility']}"
    )

# ------------------------------------------------------------
# STEP 5 — Display NDVI satellite thumbnails via GEE
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("STEP 5 — Satellite Thumbnails from GEE")
print("=" * 60)

from IPython.display import Image, display

for mine_name, info in mine_locations.items():
    lat = info['lat']
    lon = info['lon']

    point  = ee.Geometry.Point([lon, lat])
    area   = point.buffer(8000)

    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(area)
        .filterDate('2023-01-01', '2024-12-31')
        .filter(
            ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)
        )
        .median()
    )

    # RGB thumbnail
    rgb_url = s2.select(['B4','B3','B2']).getThumbURL({
        'min': 0,
        'max': 3000,
        'dimensions': 400,
        'region': area,
        'format': 'png'
    })

    # NDVI thumbnail
    ndvi_img = s2.normalizedDifference(['B8','B4'])
    ndvi_url = ndvi_img.getThumbURL({
        'min': -0.2,
        'max': 0.6,
        'dimensions': 400,
        'region': area,
        'palette': ['#d73027','#fee08b',
                    '#1a9850','#006837'],
        'format': 'png'
    })

    ndvi_val = ndvi_results[mine_name]['ndvi']
    plaus    = ndvi_results[mine_name]['plausibility']

    print(f"\n{mine_name}")
    print(f"NDVI: {ndvi_val} | Status: {plaus}")
    print("RGB (True Color):")
    display(Image(url=rgb_url, width=350))
    print("NDVI Map:")
    display(Image(url=ndvi_url, width=350))

---
## Section 9 — Summary and Conclusions

Final summary of all plausibility findings across the 5 supply chain paths combining geological validation, geographic feasibility, network structure completeness, and remote sensing confirmation.

In [ ]:
# ============================================================
# SECTION 9 — SUMMARY AND CONCLUSIONS
# Combining all plausibility findings
# ============================================================
import pandas as pd
import plotly.graph_objects as go

print("=" * 60)
print("FINAL SUMMARY — LITHIUM SUPPLY CHAIN")
print("PLAUSIBILITY ASSESSMENT")
print("=" * 60)

# FIX: Define logical supply chain order to prevent alphabetical sorting errors
tier_order = ['Mining', 'Processing', 'Manufacturing']
master['tier'] = pd.Categorical(master['tier'], categories=tier_order, ordered=True)

# Path names from master table
path_names = {}
for path_id in master['path_id'].unique():
    path_nodes = master[
        master['path_id'] == path_id
    ].sort_values('tier')

    nodes = path_nodes['node_name'].tolist()
    path_names[path_id] = (
        f"{nodes[0].split()[0]} → "
        f"{nodes[1].split()[0]} → "
        f"{nodes[2].split()[0]}"
    )

# Combine all rule scores
summary = []
for path_id in sorted(master['path_id'].unique()):
    # Rule 1 — Geology
    r1 = rule1_scores[path_id]['score']
    # Rule 2 — Distance
    r2 = rule2_scores[path_id]['score']
    # Rule 3 — Structure
    r3 = rule3_scores[path_id]['score']

    # Rule 4 — Remote sensing NDVI
    mine_name = master[
        (master['path_id'] == path_id) &
        (master['tier'] == 'Mining')
    ]['node_name'].values[0]

    ndvi_info = ndvi_results.get(mine_name, {})
    plaus = ndvi_info.get('plausibility', 'UNCERTAIN')
    ndvi_val = ndvi_info.get('ndvi', None)

    if plaus == 'CONFIRMED':
        r4 = 1.0
    elif plaus == 'CONSISTENT':
        r4 = 0.7
    else:
        r4 = 0.4

    overall = round((r1 + r2 + r3 + r4) / 4, 2)

    if overall >= 0.8:
        verdict = 'PASS'
    elif overall >= 0.5:
        verdict = 'WARN'
    else:
        verdict = 'FAIL'

    summary.append({
        'path_id':    path_id,
        'path':       path_names[path_id],
        'geology':    r1,
        'distance':   r2,
        'structure':  r3,
        'ndvi':       r4,
        'ndvi_value': ndvi_val,
        'ndvi_status': plaus,
        'overall':    overall,
        'verdict':    verdict
    })

summary_df = pd.DataFrame(summary)

print("\nFull plausibility results:")
print(summary_df[[
    'path_id', 'path',
    'geology', 'distance',
    'structure', 'ndvi',
    'overall', 'verdict']].to_string(index=False))

# ------------------------------------------------------------
# Final visualization — heatmap style results table
# ------------------------------------------------------------
verdict_colors = {
    'PASS': '#4CAF50',
    'WARN': '#FF9800',
    'FAIL': '#F44336'
}

fig = go.Figure()

# Score columns
columns = [    'geology', 'distance', 'structure', 'ndvi', 'overall']
col_labels = [    'Rule 1<br>Geology',    'Rule 2<br>Distance',    'Rule 3<br>Structure',    'Rule 4<br>NDVI',    'Overall<br>Score']

# Cell colors based on score
cell_colors = []
for col in columns:
    col_colors = []
    for val in summary_df[col]:
        if val >= 0.8:
            col_colors.append('#1B5E20')
        elif val >= 0.5:
            col_colors.append('#E65100')
        else:
            col_colors.append('#B71C1C')
    cell_colors.append(col_colors)

# Verdict column colors
verdict_cell_colors = [
    verdict_colors.get(v, '#9E9E9E')
    for v in summary_df['verdict']
]

fig.add_trace(go.Table(
    header=dict(
        values=[
            '<b>Path</b>',
            '<b>Rule 1<br>Geology</b>',
            '<b>Rule 2<br>Distance</b>',
            '<b>Rule 3<br>Structure</b>',
            '<b>Rule 4<br>NDVI</b>',
            '<b>Overall</b>',
            '<b>Verdict</b>'
        ],
        fill_color='#2a2a3e',
        font=dict(color='white', size=11),
        align='center',
        height=40
    ),
    cells=dict(
        values=[
            summary_df['path'].tolist(),
            summary_df['geology'].tolist(),
            summary_df['distance'].tolist(),
            summary_df['structure'].tolist(),
            summary_df['ndvi'].tolist(),
            summary_df['overall'].tolist(),
            summary_df['verdict'].tolist()
        ],
        fill_color=[
            ['#1a1a2e'] * len(summary_df),
            cell_colors[0],
            cell_colors[1],
            cell_colors[2],
            cell_colors[3],
            cell_colors[4],
            verdict_cell_colors
        ],
        font=dict(color='white', size=11),
        align='center',
        height=35
    )))

fig.update_layout(
    title=dict(
        text='Lithium Supply Chain — Final Plausibility Report<br>'
             '<sub>Sources: USGS MRDS | GA Critical Minerals 2025 | '
             'OSM Nominatim | Annual Reports | '
             'Sentinel-2 via GEE</sub>',
        x=0.5,
        font=dict(color='white', size=13)
    ),
    paper_bgcolor='#1a1a2e',
    height=400,
    margin=dict(l=20, r=20, t=100, b=20))

# Save
summary_path = OUTPUT_FOLDER + 'final_plausibility_report.html'
fig.write_html(summary_path)
fig.show()

print(f"\nFinal report saved to: {summary_path}")
print("\n")

print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

pass_count = len(summary_df[summary_df['verdict']=='PASS'])
warn_count = len(summary_df[summary_df['verdict']=='WARN'])
fail_count = len(summary_df[summary_df['verdict']=='FAIL'])

print(f"\nTotal paths assessed: {len(summary_df)}")
print(f"PASS: {pass_count}")
print(f"WARN: {warn_count}")
print(f"FAIL: {fail_count}")

print("\nNotable findings:")
print("1. Greenbushes NDVI (0.51) is UNCERTAIN —")
print("   mine is surrounded by dense forest in WA")
print("   requiring finer resolution validation")
print("2. Salar de Atacama NDVI (0.07) confirms")
print("   active brine extraction in Atacama desert")
print("3. China dominates processing — exports 53x")
print("   more lithium hydroxide than Australia")
print("4. Lithium price collapsed 76% from 2022-2024")
print("   creating financial stress across multiple paths")
print("5. Open Supply Hub has zero lithium industry")
print("   records — major transparency gap confirmed")
print("\n")

print("=" * 60)
print("Notebook complete")
print("All outputs saved to Google Drive outputs folder")
print("=" * 60)

In [ ]:
# ============================================================
# FIX — Recalculate Section 7 chart with all 4 rules
# Now that NDVI results are available from Section 8
# This ensures consistency between Section 7 and Section 9
# ============================================================

print("=" * 60)
print("UPDATED PLAUSIBILITY CHART — All 4 Rules")
print("Recalculated now that NDVI data is available")
print("=" * 60)

# Recalculate all scores using 4 rules
updated_results = []

for path_id in sorted(master['path_id'].unique()):

    # Rule 1 — Geology
    r1 = rule1_scores[path_id]['score']

    # Rule 2 — Distance
    r2 = rule2_scores[path_id]['score']

    # Rule 3 — Structure
    r3 = rule3_scores[path_id]['score']

    # Rule 4 — NDVI from Section 8
    mine_name = master[
        (master['path_id'] == path_id) &
        (master['tier'] == 'Mining')
    ]['node_name'].values[0]

    ndvi_info = ndvi_results.get(mine_name, {})
    plaus = ndvi_info.get('plausibility', 'UNCERTAIN')

    if plaus == 'CONFIRMED':
        r4 = 1.0
    elif plaus == 'CONSISTENT':
        r4 = 0.7
    else:
        r4 = 0.4

    overall = round((r1 + r2 + r3 + r4) / 4, 2)

    if overall >= 0.8:
        verdict = 'PASS'
    elif overall >= 0.5:
        verdict = 'WARN'
    else:
        verdict = 'FAIL'

    updated_results.append({
        'path_id':   path_id,
        'path':      path_names[path_id],
        'geology':   r1,
        'distance':  r2,
        'structure': r3,
        'ndvi':      r4,
        'overall':   overall,
        'verdict':   verdict
    })

    print(f"Path {path_id}: "
          f"G={r1} D={r2} S={r3} N={r4} "
          f"→ Overall={overall} → {verdict}")

updated_df = pd.DataFrame(updated_results)

# Rebuild chart with all 4 rules
fig_updated = go.Figure()

rules_updated = [
    ('geology',   'Rule 1: Geology'),
    ('distance',  'Rule 2: Distance'),
    ('structure', 'Rule 3: Structure'),
    ('ndvi',      'Rule 4: NDVI')
]

rule_colors = [
    '#2196F3', '#9C27B0',
    '#FF5722', '#4CAF50'
]

for (col, label), color in zip(
    rules_updated, rule_colors
):
    fig_updated.add_trace(go.Bar(
        name=label,
        x=updated_df['path'].tolist(),
        y=updated_df[col].tolist(),
        marker_color=color,
        opacity=0.85
    ))

# Overall score line
fig_updated.add_trace(go.Scatter(
    name='Overall Score',
    x=updated_df['path'].tolist(),
    y=updated_df['overall'].tolist(),
    mode='lines+markers+text',
    line=dict(color='white', width=2),
    marker=dict(size=10, color='white'),
    text=[
        f"{v}<br>{s}"
        for v, s in zip(
            updated_df['verdict'],
            updated_df['overall']
        )
    ],
    textposition='top center',
    textfont=dict(size=10, color='white')
))

# Threshold lines
fig_updated.add_hline(
    y=0.8,
    line_dash='dash',
    line_color='#4CAF50',
    annotation_text='PASS threshold (0.8)',
    annotation_position='right'
)
fig_updated.add_hline(
    y=0.5,
    line_dash='dash',
    line_color='#FF9800',
    annotation_text='WARN threshold (0.5)',
    annotation_position='right'
)

fig_updated.update_layout(
    title=dict(
        text='Plausibility Assessment — All 4 Rules<br>'
             '<sub>Rule 1: Geology | Rule 2: Distance | '
             'Rule 3: Structure | Rule 4: NDVI (Sentinel-2)'
             '</sub>',
        x=0.5,
        font=dict(color='white', size=13)
    ),
    barmode='group',
    paper_bgcolor='#1a1a2e',
    plot_bgcolor='#1a1a2e',
    font=dict(color='white'),
    xaxis=dict(
        tickangle=-15,
        gridcolor='#2a2a3e'
    ),
    yaxis=dict(
        range=[0, 1.3],
        title='Score',
        gridcolor='#2a2a3e'
    ),
    legend=dict(
        bgcolor='rgba(40,40,60,0.9)',
        font=dict(color='white')
    ),
    height=550,
    margin=dict(l=40, r=200, t=100, b=80)
)

# Save
updated_path = OUTPUT_FOLDER + 'plausibility_4rules.html'
fig_updated.write_html(updated_path)
fig_updated.show()

print(f"\nUpdated chart saved to: {updated_path}")
print("\nScores are now consistent across")
print("Section 7 and Section 9")